# DeepFilterNet3 fine-tuning with HKNT on Kaggle T4×2

Private SIH audio data is mounted from the Kaggle input `anc-sih-training-data`. The upstream DeepFilterNet3 pretrained checkpoint is the initialization/reference. The exact train/validation manifests are bundled in the private archive. This notebook runs three full training epochs, uses HKNT's `HKTrainer` optimizer/checkpoint container, exports `model.hk`, and saves a validation report. Kaggle accelerator must be set to GPU T4 x2 and Internet enabled.

In [ ]:
# === SETUP: Build deepfilterlib from source (Rust + maturin) for Python 3.12 ===
import subprocess, sys, os, types, shutil
from pathlib import Path

def sh(*args, check=True, env=None, cwd=None):
    """Run shell command, print output live, raise on failure."""
    full_env = os.environ.copy()
    full_env["PATH"] = (
        os.path.expanduser("~/.cargo/bin") + os.pathsep + full_env.get("PATH","")
    )
    if env: full_env.update(env)
    r = subprocess.run(list(args), capture_output=True, text=True,
                       env=full_env, cwd=cwd)
    out = (r.stdout + r.stderr).strip()
    if out: print(out[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"Command {args[0]} failed (exit {r.returncode})")
    return r

# ── 0. Verify GPU ──
import torch
assert torch.cuda.is_available(), "Enable GPU (T4 x2) in session settings before running"
print(f"GPU: {torch.cuda.get_device_name(0)} | count: {torch.cuda.device_count()}")

# ── 1. Ensure Rust/Cargo is available ──
print("\n[1/6] Rust toolchain...")
cargo_path = shutil.which("cargo") or os.path.expanduser("~/.cargo/bin/cargo")
if not os.path.isfile(cargo_path):
    print("  Installing rustup...")
    sh("bash", "-c",
       "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs "
       "| sh -s -- -y --default-toolchain stable --profile minimal",
       check=True)
os.environ["PATH"] = (
    os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ.get("PATH", "")
)
sh("cargo", "--version")
sh("rustup", "--version")
print("  Rust: OK")

# ── 2. Install maturin (use latest stable 1.x; we will patch pyproject.toml) ──
print("\n[2/6] Installing maturin...")
sh(sys.executable, "-m", "pip", "-q", "install", "maturin>=1.0,<2.0")
maturin_bin = shutil.which("maturin") or str(Path(sys.executable).parent / "maturin")
assert os.path.isfile(maturin_bin), f"maturin not found after install: {maturin_bin}"
sh(maturin_bin, "--version")
print("  maturin: OK")

# ── 3. Clone DeepFilterNet v0.5.6 ──
print("\n[3/6] Cloning DeepFilterNet v0.5.6 source...")
SRC = Path("/kaggle/temp/DeepFilterNet-src")
if not (SRC / "Cargo.toml").exists():
    sh("git", "clone", "--depth=1", "--branch=v0.5.6",
       "https://github.com/Rikorose/DeepFilterNet.git", str(SRC))
PYDF = SRC / "pyDF"
print(f"  Source: {SRC}")

# ── 4. Patch pyDF/pyproject.toml to accept modern maturin ──
print("\n[4/6] Patching pyDF/pyproject.toml for maturin>=1.0...")
pptoml = PYDF / "pyproject.toml"
original = pptoml.read_text()
patched = original.replace(
    'requires = ["maturin>=0.11,<0.12"]',
    'requires = ["maturin>=1.0,<2.0"]'
).replace(
    'requires = ["maturin>=0.11,<1.0"]',
    'requires = ["maturin>=1.0,<2.0"]'
)
pptoml.write_text(patched)
print("  pyproject.toml patched:")
for line in patched.split("\n"):
    if "maturin" in line: print("   ", line)

# ── 5. Build libdf wheel ──
print("\n[5/6] Building deepfilterlib (Rust → cp312 wheel, ~5-8 min)...")
WHEEL_OUT = Path("/kaggle/temp/df_wheels")
WHEEL_OUT.mkdir(parents=True, exist_ok=True)

sh(maturin_bin, "build", "--release", "--strip",
   "--interpreter", sys.executable,
   "--out", str(WHEEL_OUT),
   cwd=str(PYDF))

# Also search workspace-level target dir in case maturin ignored --out
all_search = list(WHEEL_OUT.glob("*.whl")) + list(SRC.rglob("target/wheels/*.whl"))
wheels = sorted(set(str(w) for w in all_search))
assert wheels, (
    f"No wheels found in {WHEEL_OUT} or {SRC}/target/wheels/.\n"
    f"  WHEEL_OUT contents: {list(WHEEL_OUT.iterdir())}\n"
    f"  SRC/target contents: {list((SRC/'target').glob('**/*.whl'))[:10]}"
)
wheel = wheels[-1]
print(f"  Built wheel: {Path(wheel).name}")
sh(sys.executable, "-m", "pip", "-q", "install",
   wheel, "--force-reinstall", "--no-deps")
print("  deepfilterlib: INSTALLED")

# Verify it imports
sh(sys.executable, "-c", "from libdf import DF, erb, erb_norm, unit_norm; print('libdf import OK')")

# ── 6. Install remaining Python deps ──
print("\n[6/6] Installing remaining packages...")
sh(sys.executable, "-m", "pip", "-q", "install",
   "deepfilternet==0.5.6", "--no-deps")
sh(sys.executable, "-m", "pip", "-q", "install",
   "hknt==1.1.0", "soundfile", "scipy",
   "appdirs", "loguru", "sympy",
   "packaging>=23.0,<24.0", "requests")

# torchaudio compat shim
try:
    import torchaudio
    if not hasattr(getattr(torchaudio, "backend", None), "common"):
        cm = types.ModuleType("torchaudio.backend.common")
        class AudioMetaData:
            def __init__(self, sr, nf, nc, bps, enc):
                self.sample_rate=sr; self.num_frames=nf
                self.num_channels=nc; self.bits_per_sample=bps; self.encoding=enc
        cm.AudioMetaData = AudioMetaData
        sys.modules["torchaudio.backend.common"] = cm
        if not hasattr(torchaudio, "backend"):
            bm = types.ModuleType("torchaudio.backend")
            bm.common = cm
            sys.modules["torchaudio.backend"] = bm
            torchaudio.backend = bm
        else:
            torchaudio.backend.common = cm
    print("torchaudio compat shim: OK")
except Exception as e:
    print(f"torchaudio shim note: {e}")

# ── Core imports ──
import os, tarfile, csv, random, json as _json, math, time
from pathlib import Path
import numpy as np, soundfile as sf, scipy.signal as signal
import torch.nn as nn, torch.nn.functional as F
import hk
from hk import HKTrainer, HKTrainingArguments
from df.enhance import init_df, df_features, maybe_download_model
print(f"numpy={np.__version__} torch={torch.__version__}")
print("All imports: OK")

SEED = 26052
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ── Dataset discovery ──
ROOT = Path('/kaggle/temp/anc_sih_data'); ROOT.mkdir(parents=True, exist_ok=True)
manifest_matches = list(Path('/kaggle/input').rglob('clean_train.csv'))
if manifest_matches:
    # clean_train.csv is at .../data/clean_manifests/clean_train.csv
    clean_train_path = manifest_matches[0]
    DATA = clean_train_path.parents[2]
    print(f"Dataset discovered at {DATA}")
else:
    ARCHIVE = None
    for ext in ('*.tar.gz', '*.tgz', '*.bin', '*.tar', '*.zip'):
        cands = sorted(Path('/kaggle/input').rglob(ext))
        if cands: ARCHIVE = cands[0]; break
    assert ARCHIVE is not None, (
        "Cannot find clean_train.csv or archive in /kaggle/input: "
        + str(list(Path('/kaggle/input').rglob('*'))[:30])
    )
    print(f"Extracting: {ARCHIVE}")
    if not (ROOT / 'data/clean_manifests/clean_train.csv').exists():
        if shutil.which('pigz'):
            subprocess.run(['tar','-I','pigz','-xf',str(ARCHIVE),'-C',str(ROOT)], check=True)
        else:
            with tarfile.open(ARCHIVE,'r:*') as tf: tf.extractall(ROOT)
    DATA = ROOT

MAN = DATA / 'data/clean_manifests'
assert MAN.exists(), f"Manifest dir missing: {MAN}  DATA tree: {list(DATA.rglob('*.csv'))[:15]}"
print(f"Manifests: {[f.name for f in sorted(MAN.glob('*.csv'))]}")


# ── HK Pure Python Fallback Patch ──
import hk, hk.format, struct, torch

_orig_save_file = hk.save_file

def _safe_save_file(tensors=None, filename=None, metadata=None, state_dict=None, **kwargs):
    if tensors is None:
        tensors = state_dict
    if filename is None:
        filename = kwargs.get("path", "")
    if tensors is not None:
        tensors = {k: (v.detach().cpu() if isinstance(v, torch.Tensor) else v) for k, v in tensors.items()}
    
    if getattr(hk, "is_native_available", lambda: False)():
        try:
            return _orig_save_file(tensors=tensors, filename=filename, metadata=metadata, state_dict=state_dict, **kwargs)
        except Exception as e:
            print(f"Native hk.save_file warning ({e}), falling back to pure Python HK writer...")

    # Pure Python HK container format writer
    alignment = 128
    meta_bytes = bytearray()
    n_meta = 0
    if metadata:
        for k, v in metadata.items():
            k_enc = str(k).encode('utf-8')
            v_str = str(v)
            v_enc = v_str.encode('utf-8')
            meta_bytes += struct.pack('<H', len(k_enc)) + k_enc
            meta_bytes += b'\x01'  # string tag
            meta_bytes += struct.pack('<I', len(v_enc)) + v_enc
            n_meta += 1

    header_sz = 128
    meta_off = header_sz
    meta_sz = len(meta_bytes)
    toc_off = meta_off + meta_sz
    
    DTYPE_MAP = {
        torch.float32: hk.format.STORAGE_F32,
        torch.float16: hk.format.STORAGE_F16,
        torch.int32: hk.format.STORAGE_INT32,
        torch.int64: hk.format.STORAGE_INT64,
        torch.uint8: hk.format.STORAGE_UINT8,
        torch.int8: hk.format.STORAGE_INT8,
        torch.bool: hk.format.STORAGE_BOOL,
    }
    if hasattr(torch, 'bfloat16'):
        DTYPE_MAP[torch.bfloat16] = hk.format.STORAGE_BF16

    toc_bytes = bytearray()
    n_tensors = len(tensors)
    
    temp_toc_sz = 0
    for name, tensor in tensors.items():
        n_enc = name.encode('utf-8')
        ndim = tensor.ndim
        temp_toc_sz += 2 + len(n_enc) + 4 + 8 * ndim + 54

    raw_data_off = toc_off + temp_toc_sz
    data_off = ((raw_data_off + alignment - 1) // alignment) * alignment
    
    data_bytes = bytearray()
    curr_d_off = data_off
    
    for name, tensor in tensors.items():
        n_enc = name.encode('utf-8')
        t_contig = tensor.detach().cpu().contiguous()
        t_raw = t_contig.numpy().tobytes()
        t_sz = len(t_raw)
        
        stype = DTYPE_MAP.get(tensor.dtype, hk.format.STORAGE_F32)
        ndim = tensor.ndim
        shape = tuple(tensor.shape)
        
        toc_bytes += struct.pack('<H', len(n_enc)) + n_enc
        toc_bytes += struct.pack('4B', stype, 0, 0, ndim)
        toc_bytes += struct.pack(f'<{ndim}Q', *shape)
        toc_bytes += struct.pack('<Q Q Q Q Q Q H f', curr_d_off, t_sz, 0, 0, 0, 0, 0, 0.0)
        
        data_bytes += t_raw
        curr_d_off += t_sz

    toc_sz = len(toc_bytes)
    
    header = struct.pack(
        '<4s H H I H H Q Q Q Q Q Q Q Q Q H 38s',
        hk.format.MAGIC, 1, 0, 0, alignment, 0,
        n_tensors, n_meta, meta_off, meta_sz,
        toc_off, toc_sz, data_off, 0, 0, 1, b'\x00' * 38
    )

    with open(filename, 'wb') as f:
        f.write(header)
        f.write(meta_bytes)
        f.write(toc_bytes)
        padding_len = data_off - (toc_off + toc_sz)
        if padding_len > 0:
            f.write(b'\x00' * padding_len)
        f.write(data_bytes)

hk.save_file = _safe_save_file
print("HK safe_save_file patch applied successfully!")


In [ ]:
# === LOAD PRETRAINED DEEPFILTERNET3 + HK ROUND-TRIP ===
base = maybe_download_model('DeepFilterNet3')
model, df_state, suffix = init_df(base, log_file=None)
DF_SR = df_state.sr()  # DeepFilterNet native sample rate (48000)
print(f'DeepFilterNet3 SR={DF_SR}')
model = model.cpu()
state = model.state_dict()

OUT = Path('/kaggle/working/deepfilternet3_hknt')
OUT.mkdir(parents=True, exist_ok=True)

meta = {
    'architecture': 'DeepFilterNet3',
    'origin': 'Rikorose/DeepFilterNet',
    'sample_rate': str(DF_SR),
    'hknt': '1.1.0',
    'stage': 'pretrained reference'
}
hk.save_file(state, str(OUT / 'pretrained_reference.hk'), metadata=meta)
roundtrip = hk.load_file(str(OUT / 'pretrained_reference.hk'))
model.load_state_dict(roundtrip, strict=True)
print('HK round-trip: PASSED')

device = torch.device('cuda:0')
model = model.to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
    print(f'Using DataParallel across {torch.cuda.device_count()} GPUs')
else:
    print('Using single GPU')


In [ ]:
# === DATASET SETUP ===
def rows(path):
    with open(path, newline='') as f:
        return list(csv.DictReader(f))

clean_train  = rows(MAN / 'clean_train.csv')
noise_train  = rows(MAN / 'noise_train_v2.csv')
rir_train    = rows(MAN / 'rir_train.csv')
clean_val    = rows(MAN / 'clean_val.csv')
noise_val    = rows(MAN / 'noise_val_v2.csv')
rir_val      = rows(MAN / 'rir_val.csv')
print(f'Train clean={len(clean_train)} noise={len(noise_train)} rir={len(rir_train)}')
print(f'Val   clean={len(clean_val)}  noise={len(noise_val)}  rir={len(rir_val)}')

def resolve(row):
    p = row.get('file_path') or ''
    if p:
        if (DATA / p).exists(): return DATA / p
        if (DATA / 'data' / p).exists(): return DATA / 'data' / p
        return DATA / p
    ap = row.get('audio_path') or ''
    p1 = DATA / 'data/raw_defence/iobt_gunfire/extracted' / ap
    if p1.exists(): return p1
    p2 = DATA / 'raw_defence/iobt_gunfire/extracted' / ap
    if p2.exists(): return p2
    return p1

def read_audio(path, target_sr):
    '''Read audio file, mix to mono, resample to target_sr.'''
    x, sr = sf.read(str(path), dtype='float32', always_2d=True)
    x = x.mean(axis=1)  # mono
    if sr != target_sr:
        x = signal.resample_poly(x, target_sr, sr).astype('float32')
    return x

# Determine bare model reference
def get_nb_df():
    return model.module.nb_df if isinstance(model, nn.DataParallel) else model.nb_df

class MixSet(torch.utils.data.Dataset):
    def __init__(self, clean, noise, rir, count=None, sr=48000):
        self.clean = clean
        self.noise = noise
        self.rir = rir
        self.count = count or len(clean)
        self.sr = sr
        self.n = sr * 2  # 2-second clips at target SR

    def _load_clip(self, path):
        x = read_audio(path, self.sr)
        if len(x) < self.n:
            x = np.tile(x, math.ceil(self.n / max(1, len(x))))
        start = random.randrange(max(1, len(x) - self.n + 1))
        return x[start:start + self.n]

    def __len__(self):
        return self.count

    def __getitem__(self, i):
        c = self._load_clip(resolve(self.clean[i % len(self.clean)]))
        n = self._load_clip(resolve(self.noise[random.randrange(len(self.noise))]))

        if self.rir and random.random() < 0.30:
            try:
                h = read_audio(resolve(self.rir[random.randrange(len(self.rir))]), self.sr)
                h = h[:self.sr]
                c = signal.fftconvolve(c, h, mode='full')[:len(c)].astype('float32')
            except Exception:
                pass

        snr = random.choice([-5, 0, 5, 10, 15, 20])
        c = c / (np.sqrt(np.mean(c**2)) + 1e-8)
        n = n / (np.sqrt(np.mean(n**2)) + 1e-8)
        n = n * (10 ** (-snr / 20))
        noisy = c + n
        noisy = noisy / max(1.0, float(np.max(np.abs(noisy))))

        return {
            'clean': torch.from_numpy(c.copy()).unsqueeze(0),
            'noisy': torch.from_numpy(noisy.astype('float32')).unsqueeze(0),
        }

train_ds = MixSet(clean_train, noise_train, rir_train, count=2400, sr=DF_SR)
val_ds   = MixSet(clean_val,   noise_val,   rir_val,   count=min(256, len(clean_val)), sr=DF_SR)
print(f'Train samples: {len(train_ds)}  Val samples: {len(val_ds)}')
s = train_ds[0]
print(f'Sample keys={list(s.keys())} noisy.shape={s["noisy"].shape}')


In [ ]:
# === TRAINING: 3 EPOCHS ===

def batch_df_features(audio_batch, df_state, nb_df):
    specs, erbs, specfs = [], [], []
    for i in range(audio_batch.shape[0]):
        item = audio_batch[i]
        if item.ndim == 1:
            item = item.unsqueeze(0)
        s, e, sf = df_features(item.cpu(), df_state, nb_df)
        specs.append(s.squeeze(0))
        erbs.append(e.squeeze(0))
        specfs.append(sf.squeeze(0))
    return (
        torch.stack(specs, dim=0),
        torch.stack(erbs, dim=0),
        torch.stack(specfs, dim=0)
    )

class AudioAdapter(nn.Module):
    '''Wraps DeepFilterNet3 with a combined spectral L1 loss for HK-compatible save.'''
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, noisy_t, clean_t, df_state, device):
        nb_df = get_nb_df()
        spec, erb, specf = batch_df_features(noisy_t, df_state, nb_df)
        target, _, _     = batch_df_features(clean_t, df_state, nb_df)
        
        spec   = spec.to(device)
        erb    = erb.to(device)
        specf  = specf.to(device)
        target = target.to(device)
        
        pred = self.net(spec.clone(), erb, specf)[0]
        loss = (
            F.l1_loss(pred.float(), target.float()) +
            0.25 * F.l1_loss(
                torch.linalg.vector_norm(pred.float(), dim=-1),
                torch.linalg.vector_norm(target.float(), dim=-1)
            )
        )
        return loss

    def save_hk(self, path):
        '''Save state dict to .hk container format.'''
        net = self.net.module if isinstance(self.net, nn.DataParallel) else self.net
        cpu_state = {k: (v.detach().cpu() if isinstance(v, torch.Tensor) else v) for k, v in net.state_dict().items()}
        hk.save_file(
            cpu_state, str(path),
            metadata={
                'architecture': 'DeepFilterNet3',
                'container': 'HK',
                'hknt': '1.1.0',
                'sample_rate': str(DF_SR),
                'stage': 'sih26052_finetuned',
            }
        )
        print(f'Saved HK checkpoint: {path}')

adapter = AudioAdapter(model).to(device)
optimizer = torch.optim.AdamW(adapter.parameters(), lr=1e-5, weight_decay=1e-4)

NUM_EPOCHS = 3
BATCH_SIZE = 16
LOG_EVERY  = 25

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True
)

history = []
step = 0
adapter.train()

for epoch in range(NUM_EPOCHS):
    total_loss = 0.0; seen = 0; t0 = time.time()
    for batch in train_loader:
        noisy = batch['noisy']
        clean = batch['clean']
        optimizer.zero_grad(set_to_none=True)
        loss = adapter(noisy, clean, df_state, device)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
        optimizer.step()
        step += 1; total_loss += float(loss.detach()); seen += 1
        if step % LOG_EVERY == 0:
            elapsed = time.time() - t0
            print(f'[epoch {epoch+1}/{NUM_EPOCHS}] step={step:6d}  '
                  f'loss={total_loss/seen:.5f}  elapsed={elapsed:.0f}s', flush=True)
    epoch_rec = {'epoch': epoch+1, 'mean_train_loss': total_loss/max(1,seen), 'steps': seen}
    history.append(epoch_rec)
    adapter.save_hk(OUT / f'epoch-{epoch+1}.hk')
    print(f'=== Epoch {epoch+1} complete: {epoch_rec} ===', flush=True)

print('Training complete.', flush=True)


In [ ]:
# === EVALUATION + FINAL HK SAVE + REPORT ===

val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=8, shuffle=False, num_workers=2
)

@torch.no_grad()
def evaluate(loader):
    adapter.eval()
    total = 0.0; n = 0
    for b in loader:
        noisy = b['noisy']
        clean = b['clean']
        loss = adapter(noisy, clean, df_state, device)
        total += float(loss) * len(noisy); n += len(noisy)
    adapter.train()
    return total / max(n, 1)

val_loss = evaluate(val_loader)
print(f'Final validation loss: {val_loss:.5f}')

# Save final model
adapter.save_hk(OUT / 'model.hk')

# Verify HK round-trip on final model
loaded = hk.load_file(str(OUT / 'model.hk'))
bare = model.module if isinstance(model, nn.DataParallel) else model
bare.load_state_dict(loaded, strict=True)
print('Final model HK round-trip: PASSED')

# Training report
report = {
    'architecture': 'DeepFilterNet3',
    'container_format': 'HK (hknt 1.1.0)',
    'framework': 'PyTorch + HKNT',
    'sample_rate': DF_SR,
    'epochs': NUM_EPOCHS,
    'train_examples_per_epoch': len(train_ds),
    'validation_examples': len(val_ds),
    'final_validation_loss': val_loss,
    'history': history,
    'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'hk_roundtrip': 'strict load passed',
    'output_files': [
        str(OUT / 'pretrained_reference.hk'),
        str(OUT / 'epoch-1.hk'),
        str(OUT / 'epoch-2.hk'),
        str(OUT / 'epoch-3.hk'),
        str(OUT / 'model.hk'),
    ],
}
(OUT / 'training_report.json').write_text(_json.dumps(report, indent=2))
print(_json.dumps(report, indent=2))
print(f'\nAll outputs saved to: {OUT}')
